# M5 Forecasting Accuracy — Unified Pipeline

CSV → 特徴量 parquet → train/val/eval 分割 → LightGBM 学習 → 評価 → 提出

| ステップ | 内容 |
|----------|------|
| 0. Setup | 環境検出・データ準備 |
| 1. Phase 1 | CSV → df_features.parquet (ストリーミング) |
| 2. Phase 1.5 | Store Profiling + Target Encoding |
| 3. Phase 2 | parquet → train_X.dat, val.parquet, eval.parquet |
| 4. Training | LightGBM cat_id 別 3モデル |
| 5. Evaluation | RMSE + Feature Importance |
| 6. Submission | validation + evaluation 提出ファイル生成 |

## Step 0: Setup

In [ ]:
# ============================================================
# SETUP — Colab / Local 両対応
# ============================================================
import sys, os, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")

# pip install (Colab のみ)
if IS_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'pyarrow', 'lightgbm', 'scikit-learn'])

COMPETITION = 'm5-forecasting-accuracy'
DATA_FILES  = ['calendar.csv', 'sales_train_evaluation.csv', 'sell_prices.csv']

def has_all_files(d: Path) -> bool:
    return d.exists() and all((d / f).exists() for f in DATA_FILES)

# ============================================================
# Drive マウント (Colab のみ — 出力先に使用)
# ============================================================
_drive_ok = False
if IS_COLAB:
    _drive_ok = Path('/content/drive/MyDrive').exists()
    if not _drive_ok:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            _drive_ok = True
        except Exception as _e:
            print(f'Drive mount failed: {_e}')

# ============================================================
# DATA_DIR 検出 (CSV 読み込み元)
# ============================================================
DATA_DIR = None

if IS_COLAB:
    for c in [Path(f'/content/{COMPETITION}'), Path('/content/data')]:
        if has_all_files(c):
            DATA_DIR = c
            break
    if DATA_DIR is None and _drive_ok:
        for c in [
            Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
            Path(f'/content/drive/MyDrive/{COMPETITION}'),
        ]:
            if has_all_files(c):
                DATA_DIR = c
                break
    if DATA_DIR is None:
        # CSV がどこにもない → ダウンロード
        DATA_DIR = Path(f'/content/{COMPETITION}')
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        _token = None
        _token_paths = [
            Path('/content/drive/MyDrive/.kaggle/access_token'),
            Path('/content/drive/MyDrive/kaggle/access_token'),
        ]
        for _tp in _token_paths:
            if _tp.exists():
                _token = _tp.read_text().strip()
                break
        if _token is None:
            try:
                from google.colab import userdata
                _token = userdata.get('KAGGLE_TOKEN')
            except Exception:
                pass
        if _token is None:
            raise RuntimeError(
                'Kaggle token が見つかりません。\n'
                'Drive に .kaggle/access_token を配置するか、\n'
                'Colab Secrets に KAGGLE_TOKEN を設定してください'
            )
        import requests, zipfile
        _url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMPETITION}'
        print('Downloading...')
        with requests.get(_url, headers={'Authorization': f'Bearer {_token}'},
                          stream=True, allow_redirects=True) as _r:
            _r.raise_for_status()
            _zip = DATA_DIR / f'{COMPETITION}.zip'
            with open(_zip, 'wb') as _f:
                for _chunk in _r.iter_content(1024*1024):
                    _f.write(_chunk)
        with zipfile.ZipFile(_zip) as _z:
            _z.extractall(DATA_DIR)
        _zip.unlink()
        del _token
        print('Download complete')
else:
    for c in [
        Path(f'/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/{COMPETITION}'),
        Path('.'),
    ]:
        if has_all_files(c):
            DATA_DIR = c
            break

assert DATA_DIR is not None and has_all_files(DATA_DIR), \
    f'CSV ファイルが見つかりません: {DATA_DIR}'

# ============================================================
# OUTPUT_DIR (前処理結果の書き出し先)
# ============================================================
if IS_COLAB and _drive_ok:
    OUTPUT_DIR = Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_DIR = DATA_DIR

print(f'DATA_DIR   : {DATA_DIR}  (CSV 読み込み元)')
print(f'OUTPUT_DIR : {OUTPUT_DIR}  (出力先)')
for f in DATA_FILES:
    print(f'  {f:<40} {(DATA_DIR/f).stat().st_size/1e6:6.1f} MB')

## Step 1 (Phase 1): CSV → df_features.parquet

In [ ]:
# ============================================================
# Phase 1: CSV → df_features.parquet (ストリーミング)
# ============================================================

# --- 定数 ---
PRED_HORIZON = 28
N_DAYS       = 1941
KEEP_FROM_DAY = 1100  # メモリ不足なら 1300 に上げる
CHUNK_SIZE    = 1000
PRICES_CHUNK  = 500_000

META_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
CAL_COLS  = ['d', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
             'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
             'snap_CA', 'snap_TX', 'snap_WI']
CAT_COLS  = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
             'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

# 給料日定義
PAYDAY_DAYS = {14, 15, 16, 28, 29, 30, 31}  # 15日・末日周辺
PAYDAY_EXACT = {15, 28, 29, 30, 31}          # 給料日当日
# cat_id エンコード (sorted): FOODS=0, HOBBIES=1, HOUSEHOLD=2
FOODS_CAT_ID = 0
HOBBIES_CAT_ID = 1
HOUSEHOLD_CAT_ID = 2

# EDA Step 6: イベント消費クラスター & せっかく買い指数
EVENT_CONSUMPTION_CLUSTER = {
    'Easter': 0, 'SuperBowl': 0, 'LaborDay': 0,
    'ColumbusDay': 0, 'VeteransDay': 0,
    'IndependenceDay': 1, 'Halloween': 1, 'MemorialDay': 1,
    'Christmas': 2, 'Thanksgiving': 2,
}
EVT_CLUSTER_DEFAULT = 3
IMPULSE_BUY_INDEX = {
    'Easter': 36.7, 'LaborDay': 12.8, 'SuperBowl': 5.2,
    'ColumbusDay': 3.5, 'VeteransDay': 2.1,
    'MemorialDay': -4.3, 'IndependenceDay': -8.5, 'Halloween': -11.2,
    'Christmas': 0.0, 'Thanksgiving': 0.0,
}

OUT_PATH    = OUTPUT_DIR / 'df_features.parquet'
PRICES_PATH = DATA_DIR / 'sell_prices.csv'
SALES_PATH  = DATA_DIR / 'sales_train_evaluation.csv'
CAL_PATH    = DATA_DIR / 'calendar.csv'

# --- ユーティリティ ---
def reduce_mem(df):
    for col in df.select_dtypes('integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

def stream_item_max_price(prices_path):
    result = {}
    for chunk in pd.read_csv(prices_path, chunksize=PRICES_CHUNK,
                             usecols=['item_id', 'sell_price']):
        for item_id, price in zip(chunk['item_id'], chunk['sell_price']):
            if item_id not in result or price > result[item_id]:
                result[item_id] = price
    return result

def stream_prices_for_items(prices_path, item_ids, store_ids):
    parts = []
    for chunk in pd.read_csv(prices_path, chunksize=PRICES_CHUNK):
        mask = chunk['item_id'].isin(item_ids) & chunk['store_id'].isin(store_ids)
        filtered = chunk[mask]
        if len(filtered) > 0:
            parts.append(filtered)
    if parts:
        return reduce_mem(pd.concat(parts, ignore_index=True))
    return pd.DataFrame(columns=['store_id', 'item_id', 'wm_yr_wk', 'sell_price'])

def process_chunk(chunk, d_cols, calendar, prices_path, item_max_price, keep_from_day):
    mat = chunk[d_cols].values
    has_sale = (mat > 0)
    first_idx = has_sale.argmax(axis=1)
    never_sold_mask = has_sale.sum(axis=1) == 0
    fsd = pd.DataFrame({
        'id': chunk['id'].values,
        'first_sale_day': np.where(never_sold_mask, N_DAYS + 1, first_idx + 1),
    })
    del mat, has_sale, first_idx, never_sold_mask

    df = chunk.melt(id_vars=META_COLS, value_vars=d_cols, var_name='d', value_name='sales')
    df['d_num'] = df['d'].str[2:].astype('int16')
    df['sales'] = df['sales'].astype('float32')
    df = df[df['d_num'] >= keep_from_day].reset_index(drop=True)
    df = df.merge(fsd, on='id', how='left')
    df = df[df['d_num'] >= df['first_sale_day']].drop('first_sale_day', axis=1).reset_index(drop=True)
    del fsd
    if len(df) == 0:
        return df

    df = df.merge(calendar, on='d', how='left')
    prices = stream_prices_for_items(prices_path, set(df['item_id'].unique()), set(df['store_id'].unique()))
    df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
    del prices

    # === 棚なしフラグ: sell_price が欠落 = その週は物理的に未取扱い ===
    df['not_on_shelf'] = df['sell_price'].isna().astype('int8')

    df = df.sort_values(['id', 'd_num']).reset_index(drop=True)

    df['price_change_rel'] = df.groupby('id')['sell_price'].pct_change().astype('float32')
    mp = df['item_id'].map(item_max_price).values.astype('float32')
    df['discount_ratio'] = ((mp - df['sell_price'].values) / np.where(mp == 0, np.nan, mp)).astype('float32')
    del mp

    # === SNAP 交差特徴量 (カテゴリエンコード前: state_id が文字列) ===
    _snap_map = {'CA': 'snap_CA', 'TX': 'snap_TX', 'WI': 'snap_WI'}
    df['snap_active'] = np.int8(0)
    for _state, _col in _snap_map.items():
        _m = df['state_id'] == _state
        df.loc[_m, 'snap_active'] = df.loc[_m, _col].values.astype('int8')
    df['snap_wday'] = (df['snap_active'] * df['wday']).astype('int8')

    # === [EDA Step 6] イベント消費クラスター + せっかく買い指数 ===
    df['event_consumption_type'] = (
        df['event_name_1'].map(EVENT_CONSUMPTION_CLUSTER)
        .fillna(EVT_CLUSTER_DEFAULT).astype('int8')
    )
    df['impulse_buy_index'] = (
        df['event_name_1'].map(IMPULSE_BUY_INDEX)
        .fillna(0.0).astype('float32')
    )
    # === SNAP支給日からの経過日数 (州別→統合) ===
    df['days_since_snap'] = np.int16(999)
    for _state, _dss_col in [('CA', 'days_since_snap_CA'),
                              ('TX', 'days_since_snap_TX'),
                              ('WI', 'days_since_snap_WI')]:
        _m = df['state_id'] == _state
        if _m.any():
            df.loc[_m, 'days_since_snap'] = df.loc[_m, _dss_col].values
    df.drop(columns=['days_since_snap_CA', 'days_since_snap_TX',
                      'days_since_snap_WI'], inplace=True)
    # === is_snap_first_weekend ===
    df['is_snap_first_weekend'] = np.int8(0)
    for _state, _sfwe_col in [('CA', 'is_snap_first_we_CA'),
                                ('TX', 'is_snap_first_we_TX'),
                                ('WI', 'is_snap_first_we_WI')]:
        _m = df['state_id'] == _state
        if _m.any():
            df.loc[_m, 'is_snap_first_weekend'] = df.loc[_m, _sfwe_col].values
    df.drop(columns=['is_snap_first_we_CA', 'is_snap_first_we_TX',
                      'is_snap_first_we_WI'], inplace=True)
    # === CA_4 特異性フラグ ===
    df['is_CA4'] = (df['store_id'] == 'CA_4').astype('int8')
    df['CA4_x_evt_type'] = np.where(
        df['is_CA4'] == 1, df['event_consumption_type'] + 1, 0
    ).astype('int8')

    for col in CAT_COLS:
        df[col] = df[col].astype('category').cat.codes.astype('int16')
    df['is_weekend'] = (df['wday'] >= 6).astype('int8')
    df['day'] = df['date'].dt.day.astype('int8')
    df['is_month_end'] = df['date'].dt.is_month_end.astype('int8')
    df['is_month_start'] = df['date'].dt.is_month_start.astype('int8')
    df['is_christmas_nearby'] = ((df['date'].dt.month == 12) & (df['date'].dt.day.between(23, 27))).astype('int8')

    # === 給料日・SNAP月初フラグ ===
    df['payday_flag']    = df['day'].isin(PAYDAY_EXACT).astype('int8')
    _in_window           = df['day'].isin(PAYDAY_DAYS)
    df['payday_weekend'] = (_in_window & (df['is_weekend'] == 1)).astype('int8')
    df['snap_first_10d'] = ((df['snap_active'] == 1) & (df['day'] <= 10)).astype('int8')
    del _in_window

    for lag in [28, 35, 42, 56]:
        df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag).astype('float32')
    df['_s28'] = df.groupby('id')['sales'].shift(PRED_HORIZON).astype('float32')
    for win in [7, 28, 56]:
        grp = df.groupby('id')['_s28']
        df[f'roll_mean_{win}'] = grp.rolling(win, min_periods=1).mean().reset_index(level=0, drop=True).astype('float32')
        df[f'roll_std_{win}'] = grp.rolling(win, min_periods=1).std().reset_index(level=0, drop=True).astype('float32')
    # roll_median_7: 外れ値に頑健な中央値
    df['roll_median_7'] = (
        df.groupby('id')['_s28']
          .rolling(7, min_periods=1).median()
          .reset_index(level=0, drop=True).astype('float32')
    )
    # EWMA: 指数平滑移動平均 (直近トレンド捕捉)
    df['ewma_7'] = (
        df.groupby('id')['_s28']
          .ewm(span=7, adjust=False).mean()
          .reset_index(level=0, drop=True).astype('float32')
    )
    df['ewma_28'] = (
        df.groupby('id')['_s28']
          .ewm(span=28, adjust=False).mean()
          .reset_index(level=0, drop=True).astype('float32')
    )

    # === Price rolling mean (56日) — value_gap 算出用 (Step 12d) ===
    df['price_rolling_mean_56'] = (
        df.groupby('id')['sell_price']
          .rolling(56, min_periods=1).mean()
          .reset_index(level=0, drop=True).astype('float32')
    )

    # === 補充サイクル (スパイクからの経過日数) ===
    _rm7 = df['roll_mean_7'].values
    _s28_vals = df['_s28'].values
    _spike_mask = (_s28_vals > 2 * _rm7) & (_rm7 > 0)
    df['_spike_day'] = df['d_num'].where(pd.Series(_spike_mask, index=df.index)).astype('float32')
    df['_spike_day'] = df.groupby('id')['_spike_day'].ffill()
    df['days_since_spike'] = (
        df.groupby('id')['d_num'].shift(PRED_HORIZON).astype('float32')
        - df.groupby('id')['_spike_day'].shift(PRED_HORIZON)
    ).fillna(0).clip(0, 365).astype('float32')
    df.drop(columns=['_spike_day'], inplace=True)
    del _rm7, _s28_vals, _spike_mask

    _zero = (df['_s28'] == 0).astype('float32')
    df['zeros_last_28'] = _zero.groupby(df['id']).rolling(28, min_periods=1).sum().reset_index(level=0, drop=True).astype('float32')
    _mask = df['sales'] > 0
    df['_last_sale_day'] = df['d_num'].where(_mask).astype('float32')
    df['_last_sale_day'] = df.groupby('id')['_last_sale_day'].ffill()
    df['days_since_last_sale'] = (
        df.groupby('id')['d_num'].shift(PRED_HORIZON)
        - df.groupby('id')['_last_sale_day'].shift(PRED_HORIZON)
    ).astype('float32').fillna(0)
    df.drop(columns=['_s28', '_last_sale_day'], inplace=True)
    return reduce_mem(df)

# --- 実行 ---
if OUT_PATH.exists():
    pf = pq.ParquetFile(OUT_PATH)
    print(f'[SKIP] {OUT_PATH.name} が既に存在 ({pf.metadata.num_rows:,} rows)')
    del pf
else:
    t0 = time.time()
    calendar = reduce_mem(pd.read_csv(CAL_PATH, parse_dates=['date'])[CAL_COLS])
    # イベント前後3日フラグ
    calendar = calendar.sort_values('d').reset_index(drop=True)
    _has_ev = calendar['event_name_1'].notna().astype(float)
    calendar['event_nearby'] = (
        _has_ev.rolling(7, center=True, min_periods=1).max().astype('int8')
    )
    del _has_ev

    # SNAP支給日からの経過日数 (州別)
    _d_num_cal = calendar['d'].str[2:].astype('int16')
    for _state in ['CA', 'TX', 'WI']:
        _snap_col = f'snap_{_state}'
        _last_snap_day = _d_num_cal.where(calendar[_snap_col] == 1)
        _last_snap_day = _last_snap_day.ffill()
        calendar[f'days_since_snap_{_state}'] = (
            (_d_num_cal - _last_snap_day).fillna(999).clip(0, 999).astype('int16')
        )
    # is_snap_first_weekend: SNAP期間内の最初の土日 (州別)
    _is_sat_or_sun = calendar['wday'].isin({1, 2}).astype('int8')
    for _state in ['CA', 'TX', 'WI']:
        _snap_col = f'snap_{_state}'
        _dss = calendar[f'days_since_snap_{_state}']
        calendar[f'is_snap_first_we_{_state}'] = (
            (calendar[_snap_col] == 1) & (_is_sat_or_sun == 1) & (_dss <= 6)
        ).astype('int8')
    del _d_num_cal, _is_sat_or_sun

    item_max_price = stream_item_max_price(PRICES_PATH)
    header = pd.read_csv(SALES_PATH, nrows=0)
    d_cols = [c for c in header.columns if c.startswith('d_')]
    print(f'calendar: {calendar.shape}, items: {len(item_max_price)}, days: {len(d_cols)}')

    reader = pd.read_csv(SALES_PATH, chunksize=CHUNK_SIZE)
    writer = None
    total_rows = 0
    for i, chunk in enumerate(reader):
        df_chunk = process_chunk(chunk, d_cols, calendar, PRICES_PATH, item_max_price, KEEP_FROM_DAY)
        if len(df_chunk) == 0:
            continue
        table = pa.Table.from_pandas(df_chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(OUT_PATH), table.schema)
        writer.write_table(table)
        total_rows += len(df_chunk)
        print(f'  chunk {i+1:>3} | rows {total_rows:>12,} | {time.time()-t0:.0f}s')
        del df_chunk, table
    if writer:
        writer.close()
    print(f'\nPhase 1 完了: {total_rows:,} rows ({OUT_PATH.stat().st_size/1e6:.0f} MB, {(time.time()-t0)/60:.1f} min)')

## Step 2 (Phase 1.5): Store Profiling + Target Encoding

In [ ]:
# ============================================================
# Phase 1.5: Store Profiling + Target Encoding (train期間のみ)
# ============================================================
VAL_START_DAY_15 = 1886

pf = pq.ParquetFile(OUT_PATH)
existing_cols = [f.name for f in pf.schema_arrow]
new_cols = ['snap_dependency_score', 'payroll_dependency_score',
            'weekend_intensity', 'luxury_affinity_score',
            'snap_dep_interaction', 'weekend_interaction',
            'price_sensitivity_index', 'pb_ratio',
            'price_x_psi', 'snap_x_pb',
            'cat_income_elasticity', 'stockpiling_index',
            'roll_mean_56_weighted',
            'income_event_sensitivity', 'spike_hint',
            'cat_snap_sensitivity', 'cat_payday_sensitivity',
            'snap_cat_lift', 'payday_cat_lift',
            'luxury_pressure', 'luxury_pressure_x_payday',
            'weekday_density_ratio',
            'store_dept_wday_avg', 'store_dept_premium_share',
            'value_gap', 'value_gap_x_elasticity', 'deal_intensity',
            'above_price_wall', 'price_rank_in_dept']

if all(c in existing_cols for c in new_cols):
    print(f'[SKIP] 店舗プロファイリング特徴量が既に存在')
    del pf
else:
    t0 = time.time()
    n_rg = pf.metadata.num_row_groups

    # === Pass 0: PB_Ratio 用に店舗ごとの価格 P20 閾値を算出 ===
    print('[0] 店舗別 P20 価格閾値を算出中...')
    store_prices = {}
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=['store_id', 'd_num', 'sell_price']).to_pandas()
        rg_tr = rg[rg['d_num'].values < VAL_START_DAY_15]
        for sid, price in zip(rg_tr['store_id'].values, rg_tr['sell_price'].values):
            p = float(price)
            if p > 0 and not np.isnan(p):
                sid_int = int(sid)
                if sid_int not in store_prices:
                    store_prices[sid_int] = []
                store_prices[sid_int].append(p)
        del rg, rg_tr
    p20_threshold = {}
    for sid, prices_list in store_prices.items():
        p20_threshold[sid] = float(np.percentile(prices_list, 20))
    del store_prices
    print(f'  P20 thresholds: {dict(sorted(p20_threshold.items()))}')

    # === Pass 0b: item_premium_flag (dept内 Z-score > 2.0) ===
    print('[0b] プレミアム品判定 (Z>2.0)...')
    item_price_acc = {}   # item_id(encoded) → [sum, count]
    item_to_dept = {}     # item_id(encoded) → dept_id(encoded)
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=['item_id', 'dept_id', 'sell_price']).to_pandas()
        for item, dept, price in zip(rg['item_id'].values, rg['dept_id'].values,
                                      rg['sell_price'].values):
            p = float(price)
            if p > 0 and not np.isnan(p):
                ik = int(item)
                if ik not in item_price_acc:
                    item_price_acc[ik] = [0.0, 0]
                    item_to_dept[ik] = int(dept)
                item_price_acc[ik][0] += p
                item_price_acc[ik][1] += 1
        del rg
    from collections import defaultdict
    dept_item_prices = defaultdict(list)
    item_avg_price = {}
    for ik, (s, c) in item_price_acc.items():
        avg = s / c
        item_avg_price[ik] = avg
        dept_item_prices[item_to_dept[ik]].append(avg)
    del item_price_acc
    dept_stats = {}
    for dk, prices in dept_item_prices.items():
        arr = np.array(prices)
        dept_stats[dk] = (float(arr.mean()), float(arr.std()))
    del dept_item_prices
    item_premium_flag = {}
    n_premium = 0
    for ik, avg in item_avg_price.items():
        dk = item_to_dept[ik]
        mean, std = dept_stats[dk]
        z = (avg - mean) / std if std > 0 else 0
        if z > 2.0:
            item_premium_flag[ik] = 1
            n_premium += 1
    del item_avg_price, item_to_dept
    print(f'  Premium items (Z>2.0): {n_premium}')

    # === Pass 0c: item_elasticity, item_price_cv, dept_price_range (Step 12d) ===
    print('[0c] item弾力性・価格CV・dept価格レンジ算出中...')
    # item_stats[item_id] = [n, Σp, Σs, Σp², Σps]
    item_elas_stats = {}
    # dept_price_range[dept_id] = [min_price, max_price]
    dept_price_range = {}
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=['item_id', 'dept_id', 'd_num',
                                             'sell_price', 'sales']).to_pandas()
        rg_tr = rg[rg['d_num'].values < VAL_START_DAY_15]
        for item, dept, sp, s in zip(
            rg_tr['item_id'].values, rg_tr['dept_id'].values,
            rg_tr['sell_price'].values, rg_tr['sales'].values,
        ):
            p = float(sp)
            if np.isnan(p) or p <= 0:
                continue
            sf = float(s)
            ik = int(item)
            dk = int(dept)
            if ik not in item_elas_stats:
                item_elas_stats[ik] = [0, 0.0, 0.0, 0.0, 0.0]
            st = item_elas_stats[ik]
            st[0] += 1
            st[1] += p
            st[2] += sf
            st[3] += p * p
            st[4] += p * sf
            if dk not in dept_price_range:
                dept_price_range[dk] = [p, p]
            else:
                if p < dept_price_range[dk][0]:
                    dept_price_range[dk][0] = p
                if p > dept_price_range[dk][1]:
                    dept_price_range[dk][1] = p
        del rg, rg_tr

    # item_elasticity: cov(p,s)/var(p) * mean_p/(mean_s+1)
    item_elasticity = {}
    item_price_cv = {}
    for ik, (n, sp, ss, sp2, sps) in item_elas_stats.items():
        if n < 10:
            item_elasticity[ik] = 0.0
            item_price_cv[ik] = 0.0
            continue
        mean_p = sp / n
        mean_s = ss / n
        var_p = sp2 / n - mean_p ** 2
        cov_ps = sps / n - mean_p * mean_s
        # price CV
        item_price_cv[ik] = float(np.sqrt(max(var_p, 0)) / mean_p) if mean_p > 0 else 0.0
        # normalized elasticity
        if var_p > 1e-8 and mean_s > 0:
            item_elasticity[ik] = float((cov_ps / var_p) * (mean_p / (mean_s + 1)))
        else:
            item_elasticity[ik] = 0.0
    del item_elas_stats

    # Price wall thresholds (from EDA Step 12b)
    # cat_id: 0=FOODS, 1=HOBBIES, 2=HOUSEHOLD
    PRICE_WALL = {0: 5.0, 1: 8.0, 2: 10.0}
    n_volatile = sum(1 for v in item_price_cv.values() if v > 0.05)
    print(f'  item_elasticity entries: {len(item_elasticity)}')
    print(f'  item_price_cv entries: {len(item_price_cv)} (Volatile CV>0.05: {n_volatile})')
    print(f'  dept_price_range: {dict(sorted(dept_price_range.items()))}')
    print(f'  Price wall thresholds: {PRICE_WALL}')

    # === Pass 1: train期間のみで集計 ===
    print(f'[1/3] 集計中 (d_num < {VAL_START_DAY_15} のみ)...')
    te_agg = {}
    snap_agg = {}
    payroll_agg = {}
    weekend_agg = {}
    luxury_agg = {}
    psi_agg = {}
    pb_agg = {}
    cie_agg = {}
    stockpile_daily = {}
    resid_agg = {}
    # カテゴリ別SNAP/給料日感度
    cat_snap_agg = {}    # (store_id, cat_id) → [snap_sum, snap_cnt, nosnap_sum, nosnap_cnt]
    cat_payday_agg = {}  # (store_id, cat_id) → [payday_sum, payday_cnt, nopayday_sum, nopayday_cnt]
    wdr_agg = {}         # (store_id, dept_id) → [weekday_sum, wd_cnt, weekend_sum, we_cnt]
    sdw_agg = {}         # (store_id, dept_id, wday) → [sum, count] (event/SNAP除外)
    sdps_agg = {}        # (store_id, dept_id) → [premium_sales, total_sales]

    has_snap_active = 'snap_active' in existing_cols
    has_is_weekend = 'is_weekend' in existing_cols
    base_cols = ['store_id', 'dept_id', 'd_num', 'sales', 'day', 'cat_id',
                 'discount_ratio', 'sell_price', 'roll_mean_28', 'wday',
                 'event_name_1', 'item_id']
    if has_snap_active:
        base_cols.append('snap_active')
    else:
        base_cols.extend(['state_id', 'snap_CA', 'snap_TX', 'snap_WI'])
        print('    (snap_active 列なし → snap_CA/TX/WI から再計算)')
    if has_is_weekend:
        base_cols.append('is_weekend')
    else:
        base_cols.append('wday')
        print('    (is_weekend 列なし → wday から再計算)')
    read_cols = list(dict.fromkeys(base_cols))

    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=read_cols).to_pandas()
        if not has_snap_active:
            rg['snap_active'] = np.int8(0)
            rg.loc[rg['state_id'] == 0, 'snap_active'] = rg.loc[rg['state_id'] == 0, 'snap_CA'].values.astype('int8')
            rg.loc[rg['state_id'] == 1, 'snap_active'] = rg.loc[rg['state_id'] == 1, 'snap_TX'].values.astype('int8')
            rg.loc[rg['state_id'] == 2, 'snap_active'] = rg.loc[rg['state_id'] == 2, 'snap_WI'].values.astype('int8')
        if not has_is_weekend:
            rg['is_weekend'] = (rg['wday'] >= 6).astype('int8')
        mask = rg['d_num'].values < VAL_START_DAY_15
        rg_tr = rg[mask]
        # TE 集計
        for sid, did, dnum, sales in zip(
            rg_tr['store_id'].values, rg_tr['dept_id'].values,
            rg_tr['d_num'].values, rg_tr['sales'].values,
        ):
            key = (int(sid), int(did), int(dnum))
            if key in te_agg:
                te_agg[key][0] += float(sales)
                te_agg[key][1] += 1
            else:
                te_agg[key] = [float(sales), 1]
        # --- weekday_density_ratio 集計 (store × dept) ---
        for sid, did, s, is_we in zip(
            rg_tr['store_id'].values, rg_tr['dept_id'].values,
            rg_tr['sales'].values, rg_tr['is_weekend'].values,
        ):
            wdr_key = (int(sid), int(did))
            if wdr_key not in wdr_agg:
                wdr_agg[wdr_key] = [0.0, 0, 0.0, 0]
            if int(is_we) == 0:
                wdr_agg[wdr_key][0] += float(s)
                wdr_agg[wdr_key][1] += 1
            else:
                wdr_agg[wdr_key][2] += float(s)
                wdr_agg[wdr_key][3] += 1
        # --- store_dept_wday_avg 集計 (イベント日・SNAP日除外) ---
        for sid, did, s, wday_v, snap_v, ev_v in zip(
            rg_tr['store_id'].values, rg_tr['dept_id'].values,
            rg_tr['sales'].values, rg_tr['wday'].values,
            rg_tr['snap_active'].values, rg_tr['event_name_1'].values,
        ):
            if int(snap_v) == 1:
                continue
            if int(ev_v) >= 0:
                continue
            sdw_key = (int(sid), int(did), int(wday_v))
            if sdw_key not in sdw_agg:
                sdw_agg[sdw_key] = [0.0, 0]
            sdw_agg[sdw_key][0] += float(s)
            sdw_agg[sdw_key][1] += 1
        # --- store_dept_premium_share 集計 ---
        for sid, did, s, item_v in zip(
            rg_tr['store_id'].values, rg_tr['dept_id'].values,
            rg_tr['sales'].values, rg_tr['item_id'].values,
        ):
            sdps_key = (int(sid), int(did))
            if sdps_key not in sdps_agg:
                sdps_agg[sdps_key] = [0.0, 0.0]
            sf = float(s)
            sdps_agg[sdps_key][1] += sf
            if int(item_v) in item_premium_flag:
                sdps_agg[sdps_key][0] += sf
        # 指標の集計
        for sid, snap, sales, day, is_we, cat, dr, sp, dnum, rm28 in zip(
            rg_tr['store_id'].values, rg_tr['snap_active'].values,
            rg_tr['sales'].values, rg_tr['day'].values,
            rg_tr['is_weekend'].values, rg_tr['cat_id'].values,
            rg_tr['discount_ratio'].values, rg_tr['sell_price'].values,
            rg_tr['d_num'].values, rg_tr['roll_mean_28'].values,
        ):
            sid_int = int(sid)
            s = float(sales)
            cat_int = int(cat)
            # SNAP
            if sid_int not in snap_agg:
                snap_agg[sid_int] = [0.0, 0, 0.0, 0]
            if int(snap) == 1:
                snap_agg[sid_int][0] += s
                snap_agg[sid_int][1] += 1
            else:
                snap_agg[sid_int][2] += s
                snap_agg[sid_int][3] += 1
            # Payroll
            if sid_int not in payroll_agg:
                payroll_agg[sid_int] = [0.0, 0, 0.0, 0]
            payroll_agg[sid_int][2] += s
            payroll_agg[sid_int][3] += 1
            if int(day) in PAYDAY_DAYS:
                payroll_agg[sid_int][0] += s
                payroll_agg[sid_int][1] += 1
            # Weekend
            if sid_int not in weekend_agg:
                weekend_agg[sid_int] = [0.0, 0, 0.0, 0]
            if int(is_we) == 1:
                weekend_agg[sid_int][0] += s
                weekend_agg[sid_int][1] += 1
            else:
                weekend_agg[sid_int][2] += s
                weekend_agg[sid_int][3] += 1
            # Luxury (HOBBIES)
            if sid_int not in luxury_agg:
                luxury_agg[sid_int] = [0.0, 0.0]
            luxury_agg[sid_int][1] += s
            if cat_int == HOBBIES_CAT_ID:
                luxury_agg[sid_int][0] += s
            # Price Sensitivity
            dr_f = float(dr)
            if not np.isnan(dr_f):
                if sid_int not in psi_agg:
                    psi_agg[sid_int] = [0.0, 0, 0.0, 0]
                if dr_f > 0.1:
                    psi_agg[sid_int][0] += s
                    psi_agg[sid_int][1] += 1
                else:
                    psi_agg[sid_int][2] += s
                    psi_agg[sid_int][3] += 1
            # PB Ratio
            sp_f = float(sp)
            if not np.isnan(sp_f) and sp_f > 0:
                if sid_int not in pb_agg:
                    pb_agg[sid_int] = [0.0, 0.0]
                pb_agg[sid_int][1] += s
                if sp_f <= p20_threshold.get(sid_int, 0.0):
                    pb_agg[sid_int][0] += s
            # category_income_elasticity
            cie_key = (sid_int, cat_int)
            if cie_key not in cie_agg:
                cie_agg[cie_key] = [0.0, 0, 0.0, 0]
            is_event_day = (int(snap) == 1) or (int(day) in PAYDAY_DAYS)
            if is_event_day:
                cie_agg[cie_key][0] += s
                cie_agg[cie_key][1] += 1
            else:
                cie_agg[cie_key][2] += s
                cie_agg[cie_key][3] += 1
            # cat_snap: カテゴリ別SNAP感度
            sc_key = (sid_int, cat_int)
            if sc_key not in cat_snap_agg:
                cat_snap_agg[sc_key] = [0.0, 0, 0.0, 0]
            if int(snap) == 1:
                cat_snap_agg[sc_key][0] += s
                cat_snap_agg[sc_key][1] += 1
            else:
                cat_snap_agg[sc_key][2] += s
                cat_snap_agg[sc_key][3] += 1
            # cat_payday: カテゴリ別給料日感度
            if sc_key not in cat_payday_agg:
                cat_payday_agg[sc_key] = [0.0, 0, 0.0, 0]
            if int(day) in PAYDAY_DAYS:
                cat_payday_agg[sc_key][0] += s
                cat_payday_agg[sc_key][1] += 1
            else:
                cat_payday_agg[sc_key][2] += s
                cat_payday_agg[sc_key][3] += 1
            # stockpiling_index (HOUSEHOLD daily sales)
            if cat_int == HOUSEHOLD_CAT_ID:
                dnum_int = int(dnum)
                if sid_int not in stockpile_daily:
                    stockpile_daily[sid_int] = {}
                if dnum_int not in stockpile_daily[sid_int]:
                    stockpile_daily[sid_int][dnum_int] = 0.0
                stockpile_daily[sid_int][dnum_int] += s
            # 乖離率 (residual from roll_mean_28)
            rm28_f = float(rm28)
            if rm28_f > 0 and not np.isnan(rm28_f):
                resid = (s - rm28_f) / rm28_f
                if int(snap) == 1:
                    seg = 0
                elif int(day) in PAYDAY_DAYS:
                    seg = 1
                elif int(is_we) == 1:
                    seg = 2
                else:
                    seg = 3
                rkey = (sid_int, cat_int, seg)
                if rkey not in resid_agg:
                    resid_agg[rkey] = [0.0, 0]
                resid_agg[rkey][0] += resid
                resid_agg[rkey][1] += 1
        del rg, rg_tr

    te_lookup = {k: v[0] / v[1] for k, v in te_agg.items()}
    del te_agg
    print(f'  TE lookup entries: {len(te_lookup):,}')

    # --- 指標の算出 ---
    snap_dep = {}
    for sid, (s_sum, s_cnt, ns_sum, ns_cnt) in snap_agg.items():
        snap_avg = s_sum / s_cnt if s_cnt > 0 else 0.0
        nosnap_avg = ns_sum / ns_cnt if ns_cnt > 0 else 0.0
        snap_dep[sid] = (snap_avg / nosnap_avg) if nosnap_avg > 0 else 1.0
    del snap_agg

    payroll_dep = {}
    for sid, (pd_sum, pd_cnt, t_sum, t_cnt) in payroll_agg.items():
        pd_avg = pd_sum / pd_cnt if pd_cnt > 0 else 0.0
        t_avg = t_sum / t_cnt if t_cnt > 0 else 0.0
        payroll_dep[sid] = (pd_avg / t_avg) if t_avg > 0 else 1.0
    del payroll_agg

    we_intensity = {}
    for sid, (we_sum, we_cnt, wd_sum, wd_cnt) in weekend_agg.items():
        we_avg = we_sum / we_cnt if we_cnt > 0 else 0.0
        wd_avg = wd_sum / wd_cnt if wd_cnt > 0 else 0.0
        we_intensity[sid] = (we_avg / wd_avg) if wd_avg > 0 else 1.0
    del weekend_agg

    lux_affinity = {}
    for sid, (hob_sum, tot_sum) in luxury_agg.items():
        lux_affinity[sid] = (hob_sum / tot_sum) if tot_sum > 0 else 0.0
    del luxury_agg

    price_sens = {}
    for sid, (d_sum, d_cnt, n_sum, n_cnt) in psi_agg.items():
        d_avg = d_sum / d_cnt if d_cnt > 0 else 0.0
        n_avg = n_sum / n_cnt if n_cnt > 0 else 0.0
        price_sens[sid] = (d_avg / n_avg) if n_avg > 0 else 1.0
    del psi_agg

    pb_ratio = {}
    for sid, (pb_sum, tot_sum) in pb_agg.items():
        pb_ratio[sid] = (pb_sum / tot_sum) if tot_sum > 0 else 0.0
    del pb_agg

    cie_lookup = {}
    for key, (e_sum, e_cnt, n_sum, n_cnt) in cie_agg.items():
        e_avg = e_sum / e_cnt if e_cnt > 0 else 0.0
        n_avg = n_sum / n_cnt if n_cnt > 0 else 0.0
        cie_lookup[key] = (e_avg / n_avg) if n_avg > 0 else 1.0
    del cie_agg
    print(f'  CIE entries: {len(cie_lookup)} (store×cat)')

    # cat_snap_sensitivity = snap_avg / nosnap_avg (store × cat)
    cat_snap_sens = {}
    for key, (s_sum, s_cnt, ns_sum, ns_cnt) in cat_snap_agg.items():
        s_avg = s_sum / s_cnt if s_cnt > 0 else 0.0
        ns_avg = ns_sum / ns_cnt if ns_cnt > 0 else 0.0
        cat_snap_sens[key] = (s_avg / ns_avg) if ns_avg > 0 else 1.0
    del cat_snap_agg
    print(f'  Cat SNAP sens entries: {len(cat_snap_sens)}')

    # cat_payday_sensitivity = payday_avg / nopayday_avg (store × cat)
    cat_payday_sens = {}
    for key, (p_sum, p_cnt, np_sum, np_cnt) in cat_payday_agg.items():
        p_avg = p_sum / p_cnt if p_cnt > 0 else 0.0
        np_avg = np_sum / np_cnt if np_cnt > 0 else 0.0
        cat_payday_sens[key] = (p_avg / np_avg) if np_avg > 0 else 1.0
    del cat_payday_agg
    print(f'  Cat Payday sens entries: {len(cat_payday_sens)}')

    stockpile_idx = {}
    for sid, daily in stockpile_daily.items():
        if len(daily) < 56:
            stockpile_idx[sid] = 0.0
            continue
        d_sorted = sorted(daily.keys())
        vals = np.array([daily[d] for d in d_sorted], dtype='float64')
        if vals.std() < 1e-8:
            stockpile_idx[sid] = 0.0
            continue
        n = len(vals)
        if n <= 28:
            stockpile_idx[sid] = 0.0
            continue
        mean = vals.mean()
        var = vals.var()
        autocov = np.mean((vals[:n-28] - mean) * (vals[28:] - mean))
        stockpile_idx[sid] = float(autocov / var) if var > 0 else 0.0
    del stockpile_daily
    print(f'  Stockpiling index: {dict(sorted(stockpile_idx.items()))}')

    # 乖離率の平均
    resid_mean = {}
    for rkey, (rsum, rcnt) in resid_agg.items():
        resid_mean[rkey] = rsum / rcnt if rcnt > 0 else 0.0
    del resid_agg

    # weekday_density_ratio lookup
    wdr_lookup = {}
    for key, (wd_sum, wd_cnt, we_sum, we_cnt) in wdr_agg.items():
        wd_avg = wd_sum / wd_cnt if wd_cnt > 0 else 0.0
        we_avg = we_sum / we_cnt if we_cnt > 0 else 0.0
        wdr_lookup[key] = (wd_avg / we_avg) if we_avg > 0 else 1.0
    del wdr_agg
    print(f'  WDR entries: {len(wdr_lookup)}')

    # store_dept_wday_avg lookup
    sdw_lookup = {}
    for key, (s, c) in sdw_agg.items():
        sdw_lookup[key] = s / c if c > 0 else 0.0
    del sdw_agg
    print(f'  SDW entries: {len(sdw_lookup)}')

    # store_dept_premium_share lookup
    sdps_lookup = {}
    for key, (prem, total) in sdps_agg.items():
        sdps_lookup[key] = prem / total if total > 0 else 0.0
    del sdps_agg
    print(f'  SDPS entries: {len(sdps_lookup)}')

    # income_event_sensitivity
    SEG_NAMES = {0: 'SNAP', 1: 'Payday', 2: 'Weekend'}
    income_event_sens = {}
    spike_expected = {}
    all_store_ids_r = set()
    for (sid, cid, seg), val in resid_mean.items():
        all_store_ids_r.add(sid)
        spike_expected[(sid, cid, seg)] = val
    for sid in all_store_ids_r:
        seg_totals = [0.0, 0.0, 0.0]
        seg_counts = [0, 0, 0]
        for (s, c, seg), val in resid_mean.items():
            if s == sid and seg < 3:
                seg_totals[seg] += val
                seg_counts[seg] += 1
        seg_avgs = [seg_totals[i] / seg_counts[i] if seg_counts[i] > 0 else 0.0
                    for i in range(3)]
        best_seg = int(np.argmax(seg_avgs))
        income_event_sens[sid] = best_seg
    print(f'\n    Income Event Sensitivity:')
    for sid in sorted(income_event_sens.keys()):
        seg = income_event_sens[sid]
        snap_r = resid_mean.get((sid, 0, 0), 0.0)
        pay_r = resid_mean.get((sid, 0, 1), 0.0)
        we_r = resid_mean.get((sid, 0, 2), 0.0)
        print(f'      store {sid}: {SEG_NAMES[seg]:>8} '
              f'(SNAP:{snap_r:+.3f} Pay:{pay_r:+.3f} WE:{we_r:+.3f})')

    # --- クラスタリング ---
    store_ids = sorted(snap_dep.keys())
    snap_vals = np.array([snap_dep[s] for s in store_ids])
    pay_vals = np.array([payroll_dep[s] for s in store_ids])
    snap_z = (snap_vals - snap_vals.mean()) / (snap_vals.std() + 1e-8)
    pay_z = (pay_vals - pay_vals.mean()) / (pay_vals.std() + 1e-8)

    INCOME_LABELS = {0: 'Type-S', 1: 'Type-P', 2: 'Type-B'}
    store_income = {}
    print(f'\n    Store Profiling Results:')
    print(f'    {"store":>6} {"SNAP_dep":>9} {"Payroll":>8} {"Weekend":>8} '
          f'{"Luxury":>7} {"PSI":>6} {"PB%":>6} {"CIE":>5} {"Stock":>6} {"type":>8}')
    for idx, sid in enumerate(store_ids):
        sz, pz = snap_z[idx], pay_z[idx]
        if sz > 0 and sz >= pz:
            t = 0
        elif pz > 0 and pz > sz:
            t = 1
        else:
            t = 2
        store_income[sid] = t
        cie_foods = cie_lookup.get((sid, 0), 1.0)
        print(f'    {sid:>6} {snap_dep[sid]:>9.4f} {payroll_dep[sid]:>8.4f} '
              f'{we_intensity[sid]:>8.4f} {lux_affinity[sid]:>7.4f} '
              f'{price_sens[sid]:>6.3f} {pb_ratio[sid]:>6.3f} '
              f'{cie_foods:>5.3f} {stockpile_idx.get(sid,0):>6.3f} '
              f'{INCOME_LABELS[t]:>8}')

    # Pass 2: 列を追加
    print('\n[2/3] 書き出し中...')
    tmp_path = OUT_PATH.with_suffix('.tmp.parquet')
    writer = None
    for i in range(n_rg):
        rg = pf.read_row_group(i).to_pandas()
        keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['dept_id'].astype(int).values,
            (rg['d_num'].astype(int) - 28).values,
        ))
        rg['te_store_dept_lag28'] = pd.Series(keys).map(te_lookup).astype('float32').values
        del keys
        sid_int = rg['store_id'].astype(int)
        rg['snap_dependency_score'] = sid_int.map(snap_dep).astype('float32').values
        rg['payroll_dependency_score'] = sid_int.map(payroll_dep).astype('float32').values
        rg['weekend_intensity'] = sid_int.map(we_intensity).astype('float32').values
        rg['luxury_affinity_score'] = sid_int.map(lux_affinity).astype('float32').values
        rg['price_sensitivity_index'] = sid_int.map(price_sens).astype('float32').values
        rg['pb_ratio'] = sid_int.map(pb_ratio).astype('float32').values
        rg['store_income_type'] = sid_int.map(store_income).astype('int8').values
        if 'snap_active' not in rg.columns:
            rg['snap_active'] = np.int8(0)
            rg.loc[rg['state_id'] == 0, 'snap_active'] = rg.loc[rg['state_id'] == 0, 'snap_CA'].values.astype('int8')
            rg.loc[rg['state_id'] == 1, 'snap_active'] = rg.loc[rg['state_id'] == 1, 'snap_TX'].values.astype('int8')
            rg.loc[rg['state_id'] == 2, 'snap_active'] = rg.loc[rg['state_id'] == 2, 'snap_WI'].values.astype('int8')
        if 'is_weekend' not in rg.columns:
            rg['is_weekend'] = (rg['wday'] >= 6).astype('int8')
        # payday_flag フォールバック (旧 Phase 1 の parquet にない場合)
        if 'payday_flag' not in rg.columns:
            rg['payday_flag'] = rg['day'].isin(PAYDAY_EXACT).astype('int8')
        if 'payday_weekend' not in rg.columns:
            rg['payday_weekend'] = (
                rg['day'].isin(PAYDAY_DAYS) & (rg['is_weekend'] == 1)
            ).astype('int8')
        if 'snap_first_10d' not in rg.columns:
            rg['snap_first_10d'] = (
                (rg['snap_active'] == 1) & (rg['day'] <= 10)
            ).astype('int8')
        rg['snap_dep_interaction'] = (
            rg['snap_active'].values.astype('float32') * rg['snap_dependency_score'].values
        ).astype('float32')
        rg['weekend_interaction'] = (
            rg['is_weekend'].values.astype('float32') * rg['weekend_intensity'].values
        ).astype('float32')
        rg['snap_x_income'] = (
            rg['snap_active'].values.astype('int8') * 3
            + rg['store_income_type'].values.astype('int8')
        ).astype('int8')
        rg['price_x_psi'] = (
            rg['sell_price'].values.astype('float32') * rg['price_sensitivity_index'].values
        ).astype('float32')
        rg['snap_x_pb'] = (
            rg['snap_active'].values.astype('float32') * rg['pb_ratio'].values
        ).astype('float32')
        cie_keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['cat_id'].astype(int).values,
        ))
        rg['cat_income_elasticity'] = (
            pd.Series(cie_keys).map(cie_lookup).fillna(1.0).astype('float32').values
        )
        del cie_keys
        rg['stockpiling_index'] = sid_int.map(stockpile_idx).fillna(0.0).astype('float32').values
        if 'roll_mean_56' in rg.columns:
            cie_vals = rg['cat_income_elasticity'].values
            rg['roll_mean_56_weighted'] = (
                rg['roll_mean_56'].values / np.where(cie_vals == 0, 1.0, cie_vals)
            ).astype('float32')
        else:
            rg['roll_mean_56_weighted'] = np.float32(0)
        # income_event_sensitivity
        rg['income_event_sensitivity'] = sid_int.map(income_event_sens).fillna(0).astype('int8').values
        # spike_hint
        snap_vals_j = rg['snap_active'].values.astype('int8')
        is_we_vals_j = rg['is_weekend'].values.astype('int8')
        day_vals_j = rg['day'].values if 'day' in rg.columns else np.int8(0)
        store_vals_j = rg['store_id'].astype(int).values
        cat_vals_j = rg['cat_id'].astype(int).values
        spike = np.zeros(len(rg), dtype='float32')
        for j in range(len(rg)):
            sid_j, cid_j = int(store_vals_j[j]), int(cat_vals_j[j])
            if int(snap_vals_j[j]) == 1:
                spike[j] = spike_expected.get((sid_j, cid_j, 0), 0.0)
            elif int(day_vals_j[j]) in PAYDAY_DAYS:
                spike[j] = spike_expected.get((sid_j, cid_j, 1), 0.0)
            elif int(is_we_vals_j[j]) == 1:
                spike[j] = spike_expected.get((sid_j, cid_j, 2), 0.0)
        rg['spike_hint'] = spike.astype('float32')
        # cat_snap_sensitivity / cat_payday_sensitivity (store × cat)
        sc_keys = list(zip(store_vals_j, cat_vals_j))
        rg['cat_snap_sensitivity'] = (
            pd.Series(sc_keys).map(cat_snap_sens).fillna(1.0).astype('float32').values
        )
        rg['cat_payday_sensitivity'] = (
            pd.Series(sc_keys).map(cat_payday_sens).fillna(1.0).astype('float32').values
        )
        del sc_keys
        # snap_cat_lift: SNAP日 × カテゴリ別SNAP感度 (FOODS で高い)
        rg['snap_cat_lift'] = (
            rg['snap_active'].values.astype('float32') * rg['cat_snap_sensitivity'].values
        ).astype('float32')
        # payday_cat_lift: 給料日 × カテゴリ別給料日感度 (HOBBIES で高い)
        rg['payday_cat_lift'] = (
            rg['payday_flag'].values.astype('float32') * rg['cat_payday_sensitivity'].values
        ).astype('float32')
        # not_on_shelf フォールバック (旧 Phase 1 の parquet にない場合)
        if 'not_on_shelf' not in rg.columns:
            rg['not_on_shelf'] = rg['sell_price'].isna().astype('int8')
        # luxury_pressure: sell_price × payroll_dependency_score
        # 高価格 × 給料日依存度高 = 所得制約が強い店舗にとっての購入圧力
        _sp = rg['sell_price'].fillna(0).values.astype('float32')
        rg['luxury_pressure'] = (
            _sp * rg['payroll_dependency_score'].values
        ).astype('float32')
        # luxury_pressure × payday_flag: 給料日に高価格品の需要が集中する度合い
        rg['luxury_pressure_x_payday'] = (
            rg['luxury_pressure'].values * rg['payday_flag'].values.astype('float32')
        ).astype('float32')
        del _sp
        # weekday_density_ratio
        sd_keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['dept_id'].astype(int).values,
        ))
        rg['weekday_density_ratio'] = (
            pd.Series(sd_keys).map(wdr_lookup).fillna(1.0).astype('float32').values
        )
        # store_dept_wday_avg
        sdw_keys = list(zip(
            rg['store_id'].astype(int).values,
            rg['dept_id'].astype(int).values,
            rg['wday'].astype(int).values,
        ))
        rg['store_dept_wday_avg'] = (
            pd.Series(sdw_keys).map(sdw_lookup).fillna(0.0).astype('float32').values
        )
        del sdw_keys
        # store_dept_premium_share
        rg['store_dept_premium_share'] = (
            pd.Series(sd_keys).map(sdps_lookup).fillna(0.0).astype('float32').values
        )
        del sd_keys
        # === Step 12d 新特徴量 (価格弾力性・需要構造) ===
        _sp = rg['sell_price'].fillna(0).values.astype('float32')
        _item_int = rg['item_id'].astype(int).values
        _cat_int = rg['cat_id'].astype(int).values
        _dept_int = rg['dept_id'].astype(int).values

        # value_gap: (sell_price - price_rolling_mean_56) / price_rolling_mean_56
        if 'price_rolling_mean_56' in rg.columns:
            _prm56 = rg['price_rolling_mean_56'].fillna(0).values.astype('float32')
        else:
            _prm56 = _sp  # fallback: no gap
        rg['value_gap'] = np.where(
            _prm56 > 0,
            (_sp - _prm56) / (_prm56 + 1e-8),
            np.float32(0)
        ).astype('float32')

        # item_elasticity & item_price_cv lookup (vectorized)
        _elas = np.array([item_elasticity.get(int(x), 0.0) for x in _item_int], dtype='float32')
        _pcv = np.array([item_price_cv.get(int(x), 0.0) for x in _item_int], dtype='float32')
        _is_volatile = (_pcv > 0.05).astype('float32')

        # value_gap_x_elasticity: value_gap * elasticity * (volatile only)
        rg['value_gap_x_elasticity'] = (
            rg['value_gap'].values * _elas * _is_volatile
        ).astype('float32')

        # deal_intensity: max(0, -value_gap) * elasticity * snap_active * (volatile only)
        _snap_vals = rg['snap_active'].values.astype('float32') if 'snap_active' in rg.columns else np.float32(0)
        rg['deal_intensity'] = (
            np.maximum(0, -rg['value_gap'].values) * np.abs(_elas) * _snap_vals * _is_volatile
        ).astype('float32')

        # above_price_wall: 1 if sell_price > category threshold
        _wall = np.array([PRICE_WALL.get(int(c), 10.0) for c in _cat_int], dtype='float32')
        rg['above_price_wall'] = (_sp > _wall).astype('int8')

        # price_rank_in_dept: (sell_price - dept_min) / (dept_max - dept_min)
        _dmin = np.array([dept_price_range.get(int(d), [0, 1])[0] for d in _dept_int], dtype='float32')
        _dmax = np.array([dept_price_range.get(int(d), [0, 1])[1] for d in _dept_int], dtype='float32')
        _drange = _dmax - _dmin
        rg['price_rank_in_dept'] = np.where(
            _drange > 0,
            (_sp - _dmin) / (_drange + 1e-8),
            np.float32(0.5)
        ).astype('float32')

        del _sp, _item_int, _cat_int, _dept_int, _elas, _pcv, _is_volatile
        del _snap_vals, _wall, _dmin, _dmax, _drange, _prm56

                # 旧列削除
        for old_col in ['snap_uplift_store']:
            if old_col in rg.columns:
                rg.drop(columns=[old_col], inplace=True)

        table = pa.Table.from_pandas(rg, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(tmp_path), table.schema)
        writer.write_table(table)
        del rg, table
        print(f'  rg {i+1}/{n_rg}')
    del pf
    if writer:
        writer.close()

    tmp_path.replace(OUT_PATH)
    print(f'Phase 1.5 完了 ({time.time()-t0:.0f}s, {OUT_PATH.stat().st_size/1e6:.0f} MB)')

## Step 3 (Phase 2): parquet → train/val/eval 分割

In [ ]:
# ============================================================
# Phase 2: parquet → train_X.dat, train_y.dat, train_cat.dat, val.parquet, eval.parquet
# ============================================================
VAL_START_DAY  = 1886
EVAL_START_DAY = 1914

TRAIN_X_PATH   = OUTPUT_DIR / 'train_X.dat'
TRAIN_Y_PATH   = OUTPUT_DIR / 'train_y.dat'
TRAIN_CAT_PATH = OUTPUT_DIR / 'train_cat.dat'
VAL_PATH       = OUTPUT_DIR / 'val.parquet'
EVAL_PATH      = OUTPUT_DIR / 'eval.parquet'

# FEATURES list from schema
pf = pq.ParquetFile(OUT_PATH)
all_cols = [f.name for f in pf.schema_arrow]
DROP_COLS = set(META_COLS + ['d', 'date', 'sales', 'wm_yr_wk', 'd_num',
                             'snap_CA', 'snap_TX', 'snap_WI'])
FEATURES = [c for c in all_cols if c not in DROP_COLS]
TARGET = 'sales'

# --- Target 設定 (全カテゴリ raw sales + tweedie) ---
TARGET_EPS = 1.0  # unused (kept for compatibility)
RATIO_TARGET_CATS = set()  # 空 = 全カテゴリ raw target

n_rg = pf.metadata.num_row_groups

# not_on_shelf 列の有無チェック (旧 parquet 互換)
has_not_on_shelf = 'not_on_shelf' in all_cols

# cat_id マッピング
CAT_NAMES = {0: 'FOODS', 1: 'HOBBIES', 2: 'HOUSEHOLD'}

print(f'Features ({len(FEATURES)}):')
for i, f in enumerate(FEATURES):
    print(f'  {i+1:2d}. {f}')
if has_not_on_shelf:
    print(f'\n[INFO] not_on_shelf 列あり → 棚なし行を train から除外')
print(f'\n[INFO] Target: 全カテゴリ raw sales + tweedie')

# --- 古い split ファイルとの整合性チェック ---
split_files = [TRAIN_X_PATH, TRAIN_Y_PATH, TRAIN_CAT_PATH, VAL_PATH, EVAL_PATH]
if VAL_PATH.exists():
    _val_cols = set(f.name for f in pq.ParquetFile(VAL_PATH).schema_arrow)
    _expected = set(FEATURES)
    if not _expected.issubset(_val_cols):
        _missing_feats = _expected - _val_cols
        print(f'[STALE] split ファイルに不足列: {_missing_feats}')
        print('  → 古い split ファイルを削除して再生成します')
        for p in split_files:
            if p.exists():
                p.unlink()
                print(f'    deleted: {p.name}')

# 常に再生成 (stale データによる学習事故を防止)
for p in split_files:
    if p.exists():
        p.unlink()

if True:
    t0 = time.time()

    # Pass 1: count
    print('[1/3] train 行数カウント...')
    count_cols = ['d_num', 'lag_56']
    if has_not_on_shelf:
        count_cols.append('not_on_shelf')
    n_train = 0
    for i in range(n_rg):
        rg = pf.read_row_group(i, columns=count_cols).to_pandas()
        _mask = (rg['d_num'] < VAL_START_DAY) & rg['lag_56'].notna()
        if has_not_on_shelf:
            _mask = _mask & (rg['not_on_shelf'] == 0)
        n_train += int(_mask.sum())
        del rg
    print(f'  train: {n_train:,} rows, features: {len(FEATURES)}')

    X_mm = np.memmap(str(TRAIN_X_PATH), dtype='float32', mode='w+',
                     shape=(n_train, len(FEATURES)))
    y_mm = np.memmap(str(TRAIN_Y_PATH), dtype='float32', mode='w+',
                     shape=(n_train,))
    cat_mm = np.memmap(str(TRAIN_CAT_PATH), dtype='int8', mode='w+',
                       shape=(n_train,))

    # Pass 2: split
    print('[2/3] 分割中...')
    offset = 0
    val_writer = None
    eval_writer = None
    keep_cols_ve = ['id', 'cat_id', 'd_num', TARGET] + FEATURES

    for i in range(n_rg):
        rg = pf.read_row_group(i).to_pandas()

        # Train → memmap
        mask_t = (rg['d_num'] < VAL_START_DAY) & rg['lag_56'].notna()
        if has_not_on_shelf:
            mask_t = mask_t & (rg['not_on_shelf'] == 0)
        rg[FEATURES] = rg[FEATURES].fillna(0)
        tr = rg[mask_t]
        if len(tr) > 0:
            n = len(tr)
            X_mm[offset:offset+n] = tr[FEATURES].values.astype('float32')
            y_mm[offset:offset+n] = tr[TARGET].values.astype('float32')
            cat_mm[offset:offset+n] = tr['cat_id'].values.astype('int8')
            offset += n

        mask_v = (rg['d_num'] >= VAL_START_DAY) & (rg['d_num'] < EVAL_START_DAY)
        v = rg[mask_v]
        if len(v) > 0:
            tbl = pa.Table.from_pandas(v[keep_cols_ve], preserve_index=False)
            if val_writer is None:
                val_writer = pq.ParquetWriter(str(VAL_PATH), tbl.schema)
            val_writer.write_table(tbl)

        mask_e = rg['d_num'] >= EVAL_START_DAY
        e = rg[mask_e]
        if len(e) > 0:
            tbl = pa.Table.from_pandas(e[keep_cols_ve], preserve_index=False)
            if eval_writer is None:
                eval_writer = pq.ParquetWriter(str(EVAL_PATH), tbl.schema)
            eval_writer.write_table(tbl)

        del rg, tr, v, e
        print(f'  rg {i+1}/{n_rg}  train: {offset:,}')

    X_mm.flush(); y_mm.flush(); cat_mm.flush()
    if val_writer: val_writer.close()
    if eval_writer: eval_writer.close()
    del X_mm, y_mm, cat_mm

    print(f'[3/3] 完了 ({time.time()-t0:.0f}s)')
    print(f'  train_X   : {os.path.getsize(TRAIN_X_PATH)/1e6:.0f} MB')
    print(f'  train_y   : {os.path.getsize(TRAIN_Y_PATH)/1e6:.0f} MB')
    print(f'  train_cat : {os.path.getsize(TRAIN_CAT_PATH)/1e6:.0f} MB')
    print(f'  val       : {os.path.getsize(VAL_PATH)/1e6:.0f} MB')
    print(f'  eval      : {os.path.getsize(EVAL_PATH)/1e6:.0f} MB')

del pf

# ============================================================
# 出力ファイル確認
# ============================================================
n_val  = pq.ParquetFile(VAL_PATH).metadata.num_rows
n_eval = pq.ParquetFile(EVAL_PATH).metadata.num_rows

print(f'\n=== データ概要 ===')
print(f'  train : {n_train:>12,} rows × {len(FEATURES)} features (memmap)')
print(f'  val   : {n_val:>12,} rows')
print(f'  eval  : {n_eval:>12,} rows')
print(f'\n=== 出力ファイル ({OUTPUT_DIR}) ===')
for p in [OUT_PATH, TRAIN_X_PATH, TRAIN_Y_PATH, TRAIN_CAT_PATH, VAL_PATH, EVAL_PATH]:
    if p.exists():
        print(f'  {p.name:<25} {os.path.getsize(p)/1e6:>8.0f} MB')
    else:
        print(f'  {p.name:<25} MISSING')

## Step 4: Sanity Check

In [ ]:
# ============================================================
# Step 4: Sanity Check (memmap の先頭を確認)
# ============================================================
X_peek = np.memmap(str(TRAIN_X_PATH), dtype='float32', mode='r',
                   shape=(n_train, len(FEATURES)))
y_peek = np.memmap(str(TRAIN_Y_PATH), dtype='float32', mode='r',
                   shape=(n_train,))
print('X_train[:3]:')
print(pd.DataFrame(X_peek[:3], columns=FEATURES))
print(f'\ny_train[:10]: {y_peek[:10]}')
print(f'y mean: {y_peek.mean():.3f}, std: {y_peek.std():.3f}, zeros: {(y_peek==0).sum():,}/{n_train:,}')
del X_peek, y_peek

## Step 5: モデル学習

In [ ]:
# ============================================================
# Step 5: LightGBM 学習 — 2モデル分割 (FOODS / non-FOODS)
# ============================================================
import lightgbm as lgb

# Train: memmap
X_train = np.memmap(str(TRAIN_X_PATH), dtype='float32', mode='r',
                    shape=(n_train, len(FEATURES)))
y_train = np.memmap(str(TRAIN_Y_PATH), dtype='float32', mode='r',
                    shape=(n_train,))
cat_train = np.memmap(str(TRAIN_CAT_PATH), dtype='int8', mode='r',
                      shape=(n_train,))

# Val: row_group ごとに読んで concat (小さいので許容)
pf_val = pq.ParquetFile(VAL_PATH)
val_parts = []
for i in range(pf_val.metadata.num_row_groups):
    val_parts.append(pf_val.read_row_group(i).to_pandas())
df_val = pd.concat(val_parts, ignore_index=True)
del val_parts, pf_val
df_val[FEATURES] = df_val[FEATURES].fillna(0)

_device = 'gpu' if os.environ.get('COLAB_GPU') or os.path.exists('/dev/nvidia0') else 'cpu'

# === 2モデル構成: FOODS / non-FOODS ===
# FOODS: feature_fraction=0.7 で価格系特徴量の選択確率を上げる
# non-FOODS: HOBBIES + HOUSEHOLD を統合 (48K の HOUSEHOLD を保護)

MODEL_GROUPS = {
    'FOODS': {
        'cat_ids': [0],
        'lgb_params': {
            'objective'         : 'tweedie',
            'metric'            : 'rmse',
            'num_leaves'        : 127,
            'learning_rate'     : 0.03,
            'feature_fraction'  : 0.7,    # ← 0.8→0.7: 価格系特徴量の学習機会を増やす
            'bagging_fraction'  : 0.8,
            'bagging_freq'      : 1,
            'min_child_samples' : 20,
            'device'            : _device,
            'n_jobs'            : -1,
            'seed'              : 42,
            'verbosity'         : -1,
            'two_round'         : True,
            'max_bin'           : 127,
        },
        'num_boost_round': 3000,
        'early_stopping': 100,
    },
    'NON_FOODS': {
        'cat_ids': [1, 2],  # HOBBIES + HOUSEHOLD
        'lgb_params': {
            'objective'         : 'tweedie',
            'metric'            : 'rmse',
            'num_leaves'        : 127,
            'learning_rate'     : 0.03,
            'feature_fraction'  : 0.8,
            'bagging_fraction'  : 0.8,
            'bagging_freq'      : 1,
            'min_child_samples' : 20,
            'device'            : _device,
            'n_jobs'            : -1,
            'seed'              : 42,
            'verbosity'         : -1,
            'two_round'         : True,
            'max_bin'           : 127,
        },
        'num_boost_round': 2500,
        'early_stopping': 80,
    },
}

print(f'[INFO] Device: {_device}')
print(f'[INFO] 2-model split: FOODS (ff=0.7) / NON_FOODS (HOBBIES+HOUSEHOLD)')

models = {}
for group_name, cfg in MODEL_GROUPS.items():
    print(f'\n{"="*60}')
    print(f'Training: {group_name} (cat_ids={cfg["cat_ids"]})')
    print(f'{"="*60}')

    # Train mask: cat_id in group
    tr_mask = np.isin(cat_train, cfg['cat_ids'])
    n_cat = int(tr_mask.sum())
    tr_idx = np.where(tr_mask)[0]
    X_cat = X_train[tr_idx]
    y_cat = y_train[tr_idx]

    # Val mask
    val_mask = df_val['cat_id'].isin(cfg['cat_ids'])
    df_val_cat = df_val[val_mask]

    print(f'  train: {n_cat:,} rows, val: {len(df_val_cat):,} rows')
    print(f'  rounds: {cfg["num_boost_round"]}, early_stopping: {cfg["early_stopping"]}')
    print(f'  feature_fraction: {cfg["lgb_params"]["feature_fraction"]}')

    dtrain = lgb.Dataset(
        X_cat, label=y_cat,
        feature_name=FEATURES,
        free_raw_data=True,
    )
    dval = lgb.Dataset(
        df_val_cat[FEATURES].values, label=df_val_cat[TARGET].values,
        reference=dtrain,
        free_raw_data=True,
    )

    model = lgb.train(
        cfg['lgb_params'], dtrain,
        num_boost_round=cfg['num_boost_round'],
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        callbacks=[lgb.early_stopping(cfg['early_stopping']), lgb.log_evaluation(100)],
    )
    models[group_name] = model
    print(f'  Best iteration: {model.best_iteration}')
    print(f'  Best val RMSE : {model.best_score["val"]["rmse"]:.4f}')

    del X_cat, y_cat, tr_idx, dtrain, dval, df_val_cat

del X_train, y_train, cat_train
print(f'\n{len(models)} モデル学習完了')


## Step 6: 評価 (RMSE + Feature Importance)

In [ ]:
# ============================================================
# Step 6: 評価 (RMSE + Feature Importance) — 2モデル構成
# ============================================================
from sklearn.metrics import root_mean_squared_error

# cat_id → model group mapping
def get_model_for_cat(cat_id):
    for gname, cfg in MODEL_GROUPS.items():
        if cat_id in cfg['cat_ids']:
            return models[gname]
    raise ValueError(f'No model for cat_id={cat_id}')

# Val 予測
pf_val = pq.ParquetFile(VAL_PATH)
all_true = []
all_pred = []
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    pred = np.zeros(len(rg), dtype='float32')
    for cat_id in CAT_NAMES:
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            model = get_model_for_cat(cat_id)
            pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
    all_true.append(rg[TARGET].values)
    all_pred.append(pred)
    del rg
del pf_val

val_true = np.concatenate(all_true)
val_pred = np.concatenate(all_pred)
del all_true, all_pred

rmse = root_mean_squared_error(val_true, val_pred)
print(f'Val RMSE (全体): {rmse:.4f}')

# cat_id 別 RMSE
pf_val = pq.ParquetFile(VAL_PATH)
cat_true = {c: [] for c in CAT_NAMES}
cat_pred = {c: [] for c in CAT_NAMES}
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    for cat_id in CAT_NAMES:
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            model = get_model_for_cat(cat_id)
            cat_true[cat_id].append(rg.loc[mask, TARGET].values)
            cat_pred[cat_id].append(np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None))
    del rg
del pf_val

for cat_id, cat_name in CAT_NAMES.items():
    t = np.concatenate(cat_true[cat_id])
    p = np.concatenate(cat_pred[cat_id])
    print(f'  {cat_name:>10}: RMSE={root_mean_squared_error(t, p):.4f} (n={len(t):,})')
del cat_true, cat_pred, val_true, val_pred

# --- Feature Importance: 2モデル + カテゴリ別 (FOODS / NON_FOODS) ---
os.makedirs(str(OUTPUT_DIR / 'figures'), exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

for idx, (gname, model) in enumerate(models.items()):
    imp = model.feature_importance('gain').astype('float64')
    fi = (
        pd.DataFrame({'feature': FEATURES, 'importance': imp})
        .sort_values('importance', ascending=False)
    )
    ax = axes[idx]
    fi.head(25).plot.barh(x='feature', y='importance', ax=ax, legend=False,
                           color='steelblue' if idx == 0 else 'coral')
    ax.set_title(f'Feature Importance — {gname} (Top 25, gain)')
    ax.invert_yaxis()
    print(f'\n--- {gname} Top 10 ---')
    print(fi.head(10).to_string(index=False))

    # Step 12d 新特徴量の順位を確認
    new_feats = ['value_gap', 'value_gap_x_elasticity', 'deal_intensity',
                 'above_price_wall', 'price_rank_in_dept', 'price_rolling_mean_56']
    print(f'\n--- {gname}: Step 12d 新特徴量の順位 ---')
    for feat in new_feats:
        if feat in fi['feature'].values:
            rank = fi.reset_index(drop=True)
            rank = rank[rank['feature'] == feat].index[0] + 1
            imp_val = fi[fi['feature'] == feat]['importance'].values[0]
            print(f'  {feat:30s}: rank={rank:>3d}, importance={imp_val:.2e}')
        else:
            print(f'  {feat:30s}: NOT FOUND')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'figures/feature_importance_2model.png'), dpi=100, bbox_inches='tight')
plt.show()


## Step 7: Submission 生成

In [ ]:
# ============================================================
# Step 7: Submission 生成 — 2モデル構成で予測 → pivot
# ============================================================

def predict_parquet_to_wide(parquet_path, suffix):
    pf = pq.ParquetFile(parquet_path)
    ids, d_nums, preds = [], [], []
    for i in range(pf.metadata.num_row_groups):
        rg = pf.read_row_group(i).to_pandas()
        rg[FEATURES] = rg[FEATURES].fillna(0)
        pred = np.zeros(len(rg), dtype='float32')
        for cat_id in CAT_NAMES:
            mask = (rg['cat_id'] == cat_id)
            if mask.any():
                model = get_model_for_cat(cat_id)
                pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
        ids.extend(rg['id'].values)
        d_nums.extend(rg['d_num'].values)
        preds.extend(pred)
        del rg
    del pf

    df_pred = pd.DataFrame({'id': ids, 'd_num': d_nums, 'pred': preds})
    del ids, d_nums, preds

    wide = df_pred.pivot(index='id', columns='d_num', values='pred')
    wide.columns = [f'F{i+1}' for i in range(PRED_HORIZON)]
    wide = wide.reset_index()
    wide['id'] = wide['id'].str.replace('_evaluation', f'_{suffix}', regex=False)
    return wide

sub_val  = predict_parquet_to_wide(VAL_PATH,  'validation')
sub_eval = predict_parquet_to_wide(EVAL_PATH, 'evaluation')

submission = pd.concat([sub_val, sub_eval], axis=0).reset_index(drop=True)
del sub_val, sub_eval

print(f'Submission shape: {submission.shape}')
print(f'Expected        : (60980, 29)')
assert submission.shape == (60980, 29), f'Shape mismatch: {submission.shape}'

OUT_SUB = str(OUTPUT_DIR / 'submission.csv')
submission.to_csv(OUT_SUB, index=False)
print(f'\nSaved: {OUT_SUB}')
print(f'Val RMSE: {rmse:.4f}')
print(f'\n=== Submission Summary ===')
print(f'Total rows: {len(submission):,} (validation: {len(submission)//2:,} + evaluation: {len(submission)//2:,})')
print(f'Columns: id + F1~F{PRED_HORIZON} ({submission.shape[1]} cols)')
print(f'F1 stats: mean={submission["F1"].mean():.2f}, std={submission["F1"].std():.2f}, min={submission["F1"].min():.2f}, max={submission["F1"].max():.2f}')
print(f'\n--- Head (5 rows) ---')
print(submission.head(5).to_string())
print(f'\n--- Tail (5 rows) ---')
print(submission.tail(5).to_string())
